# 仮説検証パイプライン

Phase 2（記号回帰）〜Phase 5（最適化）を対話的に実行し、
発見された数式を既知の公式と比較します。

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import sympy
import matplotlib.pyplot as plt

from fusou_formula.pipeline import Pipeline, PipelineConfig
from fusou_formula.data_loader import DataLoader
from fusou_formula.validators import FormulaValidator
from fusou_formula.phase5_optimize import RandomRangeSpec

## 1. データ準備

合成データで検証。実データの場合は `DataLoader().load_hougeki_with_stats()` を利用。

In [ ]:
loader = DataLoader()
dataset = loader.create_synthetic(
    n_samples=5000,
    formula_str='floor(firepower * 1.5 + 5)',
    noise_range=(0.0, 0.999),
)

df = dataset.df
target_col = dataset.target_col
feature_cols = dataset.feature_cols
print(f'{len(df)} samples, features: {feature_cols}')

## 2. パイプライン設定

In [ ]:
config = PipelineConfig(
    boundary_mode='max',
    parsimony=0.005,
    max_complexity=20,
    sr_niterations=50,    # ノートブック用に短縮
    sr_populations=20,
    max_breakpoints=3,
    optuna_trials=500,
    random_range_min=0.0,
    random_range_max=1.0,
)

pipeline = Pipeline(config)

## 3. Phase 1: 境界抽出

In [ ]:
phase1 = pipeline.run_phase(1, df, target_col, feature_cols)
print(f'Groups: {phase1.n_groups} (from {phase1.n_original} rows)')
phase1.df.head()

## 4. Phase 2: 記号回帰（PySR）

PySR で最適な数式をパレートフロント上で探索します。
※ 初回実行時は Julia のコンパイルに数分かかります。

In [ ]:
work_df = phase1.df

phase2 = pipeline.run_phase(2, work_df, target_col, feature_cols)

print(f'Pareto front: {len(phase2.pareto_front)} expressions')
for cand in phase2.pareto_front:
    print(f'  complexity={cand.complexity:3d}  loss={cand.loss:.6f}  {cand.latex}')

## 5. パレートフロントの可視化

In [ ]:
complexities = [c.complexity for c in phase2.pareto_front]
losses = [c.loss for c in phase2.pareto_front]

plt.figure(figsize=(8, 5))
plt.scatter(complexities, losses, s=50, zorder=3)
plt.plot(complexities, losses, 'k--', alpha=0.3)
plt.xlabel('Complexity')
plt.ylabel('Loss')
plt.title('Pareto Front: Complexity vs Loss')
plt.yscale('log')
plt.grid(True, alpha=0.3)

# ベスト式にラベル
best = phase2.best
plt.annotate(
    f'Best: {best.latex[:30]}',
    xy=(best.complexity, best.loss),
    xytext=(best.complexity + 2, best.loss * 2),
    arrowprops=dict(arrowstyle='->', color='red'),
    fontsize=9,
    color='red',
)
plt.tight_layout()
plt.show()

## 6. Phase 3: レジーム検出

In [ ]:
phase3 = pipeline.run_phase(3, work_df, target_col, feature_cols,
                             x_col=feature_cols[0])

print(f'Segments: {phase3.n_segments}')
for bp in phase3.breakpoints:
    print(f'  Breakpoint at {bp.variable} = {bp.value:.2f} (confidence={bp.confidence:.2f})')

# 可視化
phase3.plot_regimes()
plt.show()

## 7. Phase 4: AST 変異

ベスト式に `floor` 等のラッピングを試行します。

In [ ]:
base_expr = phase2.best.sympy_expr
print(f'Base expression: {base_expr}')

phase4 = pipeline.run_phase(4, work_df, target_col, feature_cols,
                             base_expr=base_expr, regime_result=phase3)

print(f'\nCandidates: {len(phase4)}')
for c in phase4[:5]:
    print(f'  [{c.complexity:3d}] {c.mutation_description}: {c.expr}')

## 8. Phase 5: Optuna 最適化

In [ ]:
phase5 = pipeline.run_phase(
    5, work_df, target_col, feature_cols,
    candidates=phase4, category_cols=dataset.category_cols,
)

print(f'Best loss: {phase5.best_loss:.6f}')
print(f'Best formula: {phase5.best_formula.latex}')
print(f'Parameters: {phase5.best_params}')

## 9. 既知式との比較

In [ ]:
# 既知の式: floor(firepower * 1.5 + 5)
known = sympy.sympify('floor(firepower * 1.5 + 5)')

validator = FormulaValidator(
    random_range=RandomRangeSpec(min_offset=0.0, max_offset=1.0),
)

# 発見式の精度チェック
metrics = validator.evaluate_accuracy(
    phase5.best_formula, df, target_col, feature_cols,
    phase5.best_params,
)

print(f'MAE:               {metrics.mae:.4f}')
print(f'RMSE:              {metrics.rmse:.4f}')
print(f'Interval accuracy: {metrics.interval_accuracy:.2%}')

# 構造比較
from fusou_formula.phase4_ast import CandidateFormula
comparison = validator.compare_with_known(
    phase5.best_formula, known, df, target_col, feature_cols,
    phase5.best_params,
)
print(f'\nStructural match: {comparison.structural_match}')
print(f'Similarity: {comparison.structural_similarity:.2%}')

## 10. レポートの生成

In [ ]:
# パイプライン全体で再実行してレポート
full_result = pipeline.run(
    df, target_col, feature_cols,
    category_cols=dataset.category_cols,
)

print(pipeline.report())